# 3. Experiments

In [1]:
import numpy as np
import gymnasium as gym
import PyFlyt.gym_envs
from algorithms import *
from tqdm import tqdm

### Fitted Q Iteration

In [2]:
# =========================
# 1. Data collection
# =========================
def collect_data(env, n_steps=10_000):
    data = []

    state, _ = env.reset()

    for _ in tqdm(range(n_steps), desc="Collecting data"):
        action = env.action_space.sample()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        data.append((state, action, reward, next_state))

        if done:
            state, _ = env.reset()
        else:
            state = next_state

    return data


# =========================
# 2. Policy evaluation
# =========================
def evaluate_policy(env, fqi, episodes=10, max_steps=500):
    returns = []

    for _ in tqdm(range(episodes), desc="Evaluating FQI policy"):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = fqi.predict_action(state)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            total_reward += reward
            state = next_state

            if done:
                break

        returns.append(total_reward)

    return np.mean(returns), np.std(returns)


# =========================
# 3. Random baseline
# =========================
def evaluate_random(env, episodes=10, max_steps=500):
    returns = []

    for _ in tqdm(range(episodes), desc="Evaluating random policy"):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):
            action = env.action_space.sample()

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            total_reward += reward
            state = next_state

            if done:
                break

        returns.append(total_reward)

    return np.mean(returns), np.std(returns)

In [ ]:
# =========================
# 4. Main experiment
# =========================
def run_experiment_fqi(flight_mode):
    print(f"\n=== Running FQI on flight_mode = {flight_mode} ===")

    # Create environment
    env = gym.make("PyFlyt/QuadX-Hover-v4", flight_mode=flight_mode)

    # Extract action bounds (important!)
    action_low = env.action_space.low
    action_high = env.action_space.high

    print("Action space low:", action_low)
    print("Action space high:", action_high)

    # -------------------------
    # Collect dataset
    # -------------------------
    print("Collecting data...")
    dataset = collect_data(env, n_steps=50000)

    # -------------------------
    # Instantiate your FQI
    # -------------------------
    from sklearn.ensemble import ExtraTreesRegressor

    model = ExtraTreesRegressor(n_estimators=50)

    # ⚠️ Replace with your class name if different
    fqi = FittedQIterationContinuous(
        model=model,
        gamma=0.99,
        action_low=action_low,
        action_high=action_high,
        n_action_samples=50
    )

    # -------------------------
    # Train
    # -------------------------
    print("Training FQI...")
    fqi.train(dataset)

    # -------------------------
    # Evaluate FQI
    # -------------------------
    print("Evaluating FQI policy...")
    mean_fqi, std_fqi = evaluate_policy(env, fqi)

    # -------------------------
    # Evaluate random baseline
    # -------------------------
    print("Evaluating random policy...")
    mean_rand, std_rand = evaluate_random(env)

    # -------------------------
    # Results
    # -------------------------
    print("\nResults:")
    print(f"FQI     → mean: {mean_fqi:.2f}, std: {std_fqi:.2f}")
    print(f"Random  → mean: {mean_rand:.2f}, std: {std_rand:.2f}")

    env.close()

In [4]:
for mode in [7]:#-1, 0, 4, 6, 7]:
    run_experiment_fqi(mode)


=== Running FQI on flight_mode = 7 ===
Action space low: [-3.14159265 -3.14159265 -3.14159265  0.        ]
Action space high: [3.14159265 3.14159265 3.14159265 0.8       ]
                             


Training FQI...


100%|██████████| 10/10 [1:53:09<00:00, 678.98s/it] 


Evaluating FQI policy...


Evaluating FQI policy:   0%|          | 0/10 [00:00<?, ?it/s]

...

Evaluating random policy:  70%|███████   | 7/10 [00:03<00:01,  1.82it/s]

Evaluating random policy:  80%|████████  | 8/10 [00:04<00:01,  1.82it/s]

Evaluating random policy:  90%|█████████ | 9/10 [00:04<00:00,  1.80it/s]

Evaluating random policy: 100%|██████████| 10/10 [00:05<00:00,  1.80it/s]


Results:
FQI     → mean: -235.61, std: 165.62
Random  → mean: -11.25, std: 179.02


In [7]:
# =========================
# Main experiment (SAC)
# =========================
def run_experiment_sac(flight_mode, episodes=200, eval_episodes=10):

    print(f"\n=== Running SAC on flight_mode = {flight_mode} ===")

    # -------------------------
    # Create environment
    # -------------------------
    env = gym.make("PyFlyt/QuadX-Hover-v4", flight_mode=flight_mode)

    action_low = env.action_space.low
    action_high = env.action_space.high

    print("Action space low:", action_low)
    print("Action space high:", action_high)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    # -------------------------
    # Instantiate SAC
    # -------------------------
    from algorithms import SACAgent
    agent = SACAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        action_low=action_low,
        action_high=action_high,
    )

    # =====================================================
    # TRAINING
    # =====================================================
    print("Training SAC...")

    for ep in range(episodes):

        state, _ = env.reset()
        ep_reward = 0

        for _ in range(500):

            # action in normalized space [-1, 1]
            action = agent.act(state)

            # scale to environment bounds
            env_action = agent._scale_action(
                __import__("torch").tensor(action)
            ).cpu().numpy()

            next_state, reward, terminated, truncated, _ = env.step(env_action)
            done = terminated or truncated

            agent.store(state, action, reward, next_state, done)
            agent.update()

            state = next_state
            ep_reward += reward

            if done:
                break

        print(f"Episode {ep:3d} | reward: {ep_reward:.2f}")

    # =====================================================
    # EVALUATION
    # =====================================================
    print("\nEvaluating SAC policy...")

    def evaluate(agent, env, episodes=10):
        returns = []

        for _ in range(episodes):
            state, _ = env.reset()
            total = 0

            for _ in range(500):
                action = agent.act(state, eval=True)
                env_action = agent._scale_action(
                    __import__("torch").tensor(action)
                ).cpu().numpy()

                state, reward, terminated, truncated, _ = env.step(env_action)
                total += reward

                if terminated or truncated:
                    break

            returns.append(total)

        return np.mean(returns), np.std(returns)

    mean_sac, std_sac = evaluate(agent, env, eval_episodes)

    # -------------------------
    # Random baseline
    # -------------------------
    print("Evaluating random policy...")

    def evaluate_random(env, episodes=10):
        returns = []

        for _ in range(episodes):
            state, _ = env.reset()
            total = 0

            for _ in range(500):
                action = env.action_space.sample()
                state, reward, terminated, truncated, _ = env.step(action)
                total += reward

                if terminated or truncated:
                    break

            returns.append(total)

        return np.mean(returns), np.std(returns)

    mean_rand, std_rand = evaluate_random(env, eval_episodes)

    # -------------------------
    # Results
    # -------------------------
    print("\nResults:")
    print(f"SAC     → mean: {mean_sac:.2f}, std: {std_sac:.2f}")
    print(f"Random  → mean: {mean_rand:.2f}, std: {std_rand:.2f}")

    env.close()

In [8]:
for mode in [7]:#-1, 0, 4, 6, 7]:
    run_experiment_sac(mode)


=== Running SAC on flight_mode = 7 ===
Action space low: [-3.14159265 -3.14159265 -3.14159265  0.        ]
Action space high: [3.14159265 3.14159265 3.14159265 0.8       ]


ImportError: cannot import name 'SACAgent' from 'algorithms' (c:\Users\andre\OneDrive\Bureau\ULg\M1\INFO-8003 - Reinforcement Learning\Project\Info8003-reinforcement-learning-project-2026\algorithms.py)